# Build Database (Testing-Stage)
Test building the schema, process the raw dataframes through cleaning functions and push them into a SQL database.

## Setup

In [1]:
import sys
import os
import pandas as pd
from sqlalchemy import create_engine

# Set root path
sys.path.append(os.path.abspath(".."))

from etl_pipeline.api_utils import run_batch_ingestion
from etl_pipeline.pipeline_utils import (
    generate_dim_time,
    generate_dim_date,

    clean_countries_db,
    clean_cities_db,
    clean_airports_db,
    clean_airlines_db,
    clean_aircraft_db,
    clean_flights,
    clean_schedules,

    build_fact_flight,
    load_incremental_flights,

    # enrich_dim_aircraft,
    enrich_dim_airlines,
)
from archive.pipeline_utils_OLD import enrich_dim_aircraft

## Initialising Database

In [2]:
# =========================================================================
# DATABASE CONFIGURATION
# =========================================================================
DB_NAME = "airlines_warehouse.db"
DB_PATH = f"data/{DB_NAME}"

# Ensure the directory exists
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

# Check if the database file already exists
db_exists = os.path.exists(DB_PATH)

# Create SQLAlchemy engine connection
engine = create_engine(f"sqlite:///{DB_PATH}")

# =========================================================================
# INITIALIZATION (Runs only if DB doesn't exist)
# =========================================================================
if not db_exists:
    print("Initial run detected: Creating database schema and base dimensions...")

    # Execute schema.sql file to build tables, PKs, and FKs
    with open("../database_schema/schema.sql", "r") as f:
        schema_sql = f.read()

    with engine.begin() as conn:
        # SQLite execution requires raw connection execution for multi-statement DDL scripts
        raw_conn = conn.connection
        cursor = raw_conn.cursor()
        cursor.executescript(schema_sql)
        raw_conn.commit()

    print("Schema created successfully!")

    # Generate and load static Date and Time dimensions
    dim_date = generate_dim_date()
    dim_date.to_sql('dim_date', engine, if_exists='append', index=False)

    dim_time = generate_dim_time()
    dim_time.to_sql('dim_time', engine, if_exists='append', index=False)

    print("Static time dimensions populated.")

    print("Ingesting initial datasets from AirLabs API...")
    # Define endpoints and their specific requirements
    ingestion_plan = {
        #'flights': {},
        # 'schedules': {'dep_iata': 'LHR'}, # dep_iata/dep_icao
        # 'schedules': {'arr_iata': 'LHR'}, # arr_iata/arr_icao
        'airportsDB': {},
        'airlinesDB': {},
        'citiesDB': {},
        'countriesDB': {},
        'fleetsDB': {'airline_icao': 'BAW'},
        #
        #
    }

    df = run_batch_ingestion(ingestion_plan, verbose=False)

    print("Transforming baseline datasets...")
    # Create Independent Dimensions
    dim_countries = clean_countries_db(df['countriesDB'])
    dim_cities = clean_cities_db(df['citiesDB'])
    dim_airports = clean_airports_db(df['airportsDB'])
    dim_airlines = clean_airlines_db(df['airlinesDB'])
    dim_aircrafts = clean_aircraft_db(df['fleetsDB'])
    # dim_time ?
    # dim_schedule ?

    # Create Dependent Fact Table & Telemetry/Time Dimension
    # fact_flights, dim_flight_position, df_live_aircraft_patch = clean_flights(df['flights'])

    # ================================
    # Load Dim & Fact Tables into SQL
    print("Loading dimension and fact tables into the database...")

    # Independent Dimensions first (so foreign keys are ready)
    dim_countries.to_sql('dim_country', engine, if_exists='append', index=False)
    dim_cities.to_sql('dim_city', engine, if_exists='append', index=False)
    dim_airports.to_sql('dim_airport', engine, if_exists='append', index=False)
    dim_airlines.to_sql('dim_airline', engine, if_exists='append', index=False)
    dim_aircrafts.to_sql('dim_aircraft', engine, if_exists='append', index=False)

    # Dependent Dimensions & Fact
    # fact_flights.to_sql('fact_flight', engine, if_exists='append', index=False)
    # dim_flight_position.to_sql('dim_flight_position', engine, if_exists='append', index=False)

    print("Initial setup complete! Database created and populated successfully.")

else:
    print("Existing database found. Skipping schema creation. Ready for incremental append.")


Initial run detected: Creating database schema and base dimensions...
Schema created successfully!
Static time dimensions populated.
Ingesting initial datasets from AirLabs API...
Starting Batch Ingestion at 20260723_20...
Successfully ingested airportsDB
Successfully ingested airlinesDB
Successfully ingested citiesDB
Successfully ingested countriesDB
Successfully ingested fleetsDB
Transforming baseline datasets...
Original records from /countries: 252
Unique countries for DIM_COUNTRIES: 252
Original records from /citiesDB: 10273
Records after dropping NaNs: 10273
Unique cities for DIM_CITIES: 10273
Original records from /airports: 23349
Unique airports for DIM_AIRPORT: 20878
Original records from /airlines: 6576
Count after keeping non-null 'icao_code' rows: 6259
Final unique airline records for DIM_AIRLINES: 6240
Original records from /fleets: 50
Dropped 0 because of missing airplane_hex.
Unique aircraft records for DIM_AIRCRAFT: 50
Loading dimension and fact tables into the database

In [17]:
# =========================================================================
# DAILY INCREMENTAL RUN (Runs every execution)
# =========================================================================
print("Starting routine flight & schedule ingestion...")

# Extract only fresh live flights on daily runs
daily_plan = {
    'flights': {},
    'schedules': {'dep_iata': 'LHR'}, # dep_iata/dep_icao
    'schedules': {'arr_iata': 'LHR'}, # arr_iata/arr_icao
    # 'flight'?,
    # 'schedule'?
}
df_raw = run_batch_ingestion(daily_plan, verbose=False)

if df_raw.get('flights') is not None and not df_raw['flights'].empty:
    print("Cleaning live telemetry and schedules...")

    # Clean the raw data
    fact_flights, dim_flight_position, df_live_aircraft_patch = clean_flights(df_raw['flights'])

    if df_raw.get('schedules') is not None and not df_raw['schedules'].empty:
        clean_scheds = clean_schedules(df_raw['schedules'])
    else:
        # Failsafe, if schedules endpoint returns nothing
        clean_scheds = pd.DataFrame(columns=['flight_icao'])

    # Merge: Attach schedules to the live flights
    final_fact_flights = build_fact_flight(fact_flights, clean_scheds)



    # =========================================================================
    # STATUS UPDATE SAFETY NET: Force stale 'en-route' flights to landed/timeout
    """
    if 'status' in final_fact_flights.columns:
        current_time = pd.Timestamp.now()

        # 1. Update based on passed arrival times (if available)
        if 'arrival_time' in final_fact_flights.columns:
            final_fact_flights['arrival_time'] = pd.to_datetime(final_fact_flights['arrival_time'])
            final_fact_flights.loc[
                (final_fact_flights['status'] == 'en-route') &
                (final_fact_flights['arrival_time'] <= current_time),
                'status'
            ] = 'landed'

        # 2. Safety net: Force lingering 'en-route' records past a max threshold (e.g., 16 hours)
        if 'departure_time' in final_fact_flights.columns:
            final_fact_flights['departure_time'] = pd.to_datetime(final_fact_flights['departure_time'])
            max_duration = pd.Timedelta(hours=16)

            final_fact_flights.loc[
                (final_fact_flights['status'] == 'en-route') &
                (current_time - final_fact_flights['departure_time'] > max_duration),
                'status'
            ] = 'ASSUMED_LANDED'
    """

    # Enrichment: Update the aircraft dimension with live patch
    try:
        print("Checking for new aircraft data to enrich dim_aircraft...")
        existing_aircraft = pd.read_sql("SELECT * FROM dim_aircraft;", engine)
        dim_aircraft_enriched = enrich_dim_aircraft(existing_aircraft, df_live_aircraft_patch)
        dim_aircraft_enriched.to_sql('dim_aircraft', engine, if_exists='replace', index=False)

        print("Checking live flights to patch missing Airline data...")
        existing_airlines = pd.read_sql("SELECT * FROM dim_airline;", engine)
        dim_airline_enriched = enrich_dim_airlines(existing_airlines, df_raw['flights'])
        dim_airline_enriched.to_sql('dim_airline', engine, if_exists='replace', index=False)
    except Exception as e:
        print(f"Skipping aircraft enrichment due to error: {e}")

    # =========================================================================
    # DATABASE LOAD
    print("Pushing incremental data to the data warehouse...")
    added_flights, added_telemetry = load_incremental_flights(engine, final_fact_flights, dim_flight_position)

    print("Pipeline run completed successfully!")
    print(f"Audit: Added {added_flights} new flights and {added_telemetry} telemetry points.")

else:
    print("No flight data returned from API today. Pipeline aborted.")

Starting routine flight & schedule ingestion...
Starting Batch Ingestion at 20260723_20...
Successfully ingested flights
Successfully ingested schedules
Cleaning live telemetry and schedules...
Original records from /flights: 6336
Processed 6336 raw movements into FACT_FLIGHTS and DIM_FLIGHT_POSITION, and Live Aircraft Patch.
Checking for new aircraft data to enrich dim_aircraft...
Checking live flights to patch missing Airline data...
Pushing incremental data to the data warehouse...
Added 180 new flights.
Appended 6336 new telemetry points.
Pipeline run completed successfully!
Audit: Added 180 new flights and 6336 telemetry points.


## First Look

In [18]:
import sqlite3
import pandas as pd

# Connect to your database
conn = sqlite3.connect("data/airlines_warehouse.db")

# Check what tables were created successfully
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Tables in database:")
print(tables)

# Peek at the first few rows of flights fact table
print("\nPreview of fact_flight:")
fact_preview = pd.read_sql("SELECT * FROM fact_flight WHERE airline_icao = 'BAW' LIMIT 50;", conn)
print(fact_preview)

conn.close()

Tables in database:
                  name
0          dim_country
1             dim_city
2          dim_airport
3             dim_date
4             dim_time
5  dim_flight_position
6          fact_flight
7         dim_aircraft
8          dim_airline

Preview of fact_flight:
           flight_id flight_number movement_type    status dep_delayed_min  \
0     BAW99_20260723            99          None  en-route            None   
1    BAW178_20260723           178          None  en-route            None   
2   SHT1338_20260723          1338          None  en-route            None   
3    BAW845_20260723           845          None  en-route            None   
4    BAW397_20260723           397          None  en-route            None   
5    BAW398_20260723           398          None  en-route            None   
6     BAW61_20260723            61          None  en-route            None   
7    BAW530_20260723           530          None  en-route            None   
8    BAW695_20260723   

In [19]:
import pandas as pd

# Busiest Airports Right Now
query = """
SELECT
    a.airport_name AS origin_airport,
    a.country_code,
    t.shift AS time_of_day,
    COUNT(f.flight_id) AS active_flights
FROM fact_flight f
JOIN dim_airport a ON f.origin_airport_id = a.icao_code
JOIN dim_time t ON f.updated_time_key = t.time_key
WHERE f.status = 'en-route'
GROUP BY a.airport_name, a.country_code, t.shift
ORDER BY active_flights DESC
LIMIT 10;
"""

# Execute and display!
results_df = pd.read_sql(query, engine)
print(results_df)

                                     origin_airport country_code time_of_day  \
0                      Denver International Airport           US     Evening   
1           Dallas/Fort Worth International Airport           US     Evening   
2                 Los Angeles International Airport           US     Evening   
3              Chicago O'Hare International Airport           US     Evening   
4  Hartsfield-Jackson Atlanta International Airport           US     Evening   
5          Phoenix Sky Harbor International Airport           US     Evening   
6              Seattle-Tacoma International Airport           US     Evening   
7             John F. Kennedy International Airport           US     Evening   
8               San Francisco International Airport           US     Evening   
9              George Bush Intercontinental Airport           US     Evening   

   active_flights  
0             156  
1             154  
2             146  
3             134  
4             133  

In [20]:
import pandas as pd

# Airlines Plane count (currently in the air)
query = """
SELECT
    al.airline_name,
    COUNT(f.flight_id) AS planes_in_air
FROM fact_flight f
JOIN dim_airline al ON f.airline_icao = al.icao_code
JOIN dim_aircraft ac ON f.aircraft_hex = ac.hex
WHERE f.status = 'en-route'
GROUP BY al.airline_name
ORDER BY planes_in_air DESC
LIMIT 10;
"""

# Execute and display!
results_df = pd.read_sql(query, engine)
print(results_df)

         airline_name  planes_in_air
0   American Airlines            599
1     Delta Air Lines            525
2  Southwest Airlines            478
3     United Airlines            475
4    SkyWest Airlines            239
5     Alaska Airlines            167
6             Ryanair            163
7     Jetblue Airways            157
8    Netjets Aviation            151
9    Republic Airline            119


In [22]:
import pandas as pd

# Airlines Plane count (currently in the air)
query = """
SELECT status, COUNT(*)
FROM fact_flight
GROUP BY status;
"""

# Execute and display!
results_df = pd.read_sql(query, engine)
print(results_df)




      status  COUNT(*)
0   en-route      7283
1     landed       147
2  scheduled        66


In [5]:
import sqlite3

# Connect to your warehouse database
conn = sqlite3.connect("./database/airlines_warehouse.db")
cursor = conn.cursor()

# Print the exact schema definition stored in SQLite
cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name='dim_aircraft';")
print("--- TABLE SCHEMA ---")
print(cursor.fetchone()[0])

# Print actual physical columns present in the table
cursor.execute("PRAGMA table_info(dim_aircraft);")
print("\n--- ACTUAL COLUMNS ---")
for col in cursor.fetchall():
    print(col[1])

conn.close()

OperationalError: unable to open database file